<a href="https://colab.research.google.com/github/Krishnan-Raghavan/Packt/blob/main/StableDiffusionChapter18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
from transformers import CLIPSegProcessor,CLIPSegForImageSegmentation

processor = CLIPSegProcessor.from_pretrained(
    "CIDAS/clipseg-rd64-refined"
)
model = CLIPSegForImageSegmentation.from_pretrained(
    "CIDAS/clipseg-rd64-refined"
)

In [ ]:
!pip install diffusers
!pip install transformers scipy ftfy accelerate ipywidgets

In [ ]:
from diffusers.utils import load_image
from diffusers.utils.pil_utils import numpy_to_pil
import torch

source_image = load_image("/content/clipseg_source_image.png")

prompts = ['the background']
inputs = processor(
    text             = prompts
    , images         = [source_image] * len(prompts)
    , padding        = True
    , return_tensors = "pt"
)

with torch.no_grad():
    outputs = model(**inputs)

preds = outputs.logits
mask_data = torch.sigmoid(preds)

mask_data_numpy = mask_data.detach().unsqueeze(-1).numpy()
mask_pil = numpy_to_pil(mask_data_numpy)[0].resize(source_image.size)

mask_pil

In [ ]:
bw_thresh = 100
bw_fn = lambda x : 255 if x > bw_thresh else 0
bw_mask_pil = mask_pil.convert("L").point(bw_fn, mode="1")
bw_mask_pil

In [ ]:
from diffusers import StableDiffusionInpaintPipeline, EulerDiscreteScheduler
inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4"
    , torch_dtype = torch.float16
    , safety_checker = None
).to("cuda:0")

sd_prompt = "blue sky and mountains"
out_image = inpaint_pipe(
    prompt          = sd_prompt
    , image         = source_image
    , mask_image    = bw_mask_pil
    , strength      = 0.9
    , generator     = torch.Generator("cuda:0").manual_seed(7)
).images[0]
out_image

In [ ]:
!jupyter nbconvert --execute main.ipynb

In [ ]:
!pip install accelerate

In [ ]:
bw_thresh = 100
bw_fn = lambda x : 255 if x > bw_thresh else 0
bw_mask_pil = mask_pil.convert("L").point(bw_fn, mode="1")
bw_mask_pil

In [ ]:
from PIL import Image, ImageOps
output_image = Image.new("RGBA", source_image.size, (255,255,255,255))
inverse_bw_mask_pil = ImageOps.invert(bw_mask_pil)
r = Image.composite(source_image ,output_image, inverse_bw_mask_pil)
r

In [ ]:
from rembg import remove
remove(source_image)

In [ ]:
from rembg import remove
from PIL import Image
#white_bg = Image.new("RGBA", source_image.size, (255,255,255))
black_bg = Image.new("RGBA", source_image.size, (0,0,0))
image_wo_bg = remove(source_image)
img_wo_bg = Image.alpha_composite(black_bg, image_wo_bg)
img_wo_bg

In [ ]:
import torch
from transformers import CLIPVisionModelWithProjection
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "h94/IP-Adapter",
    subfolder   = "models/image_encoder",
    torch_dtype = torch.float16,
).to("cuda:0")

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline
pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    image_encoder   = image_encoder,
    torch_dtype     = torch.float16,
    safety_checker  = None
).to("cuda:0")

In [ ]:
pipeline.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder       = "models",
    weight_name     = "ip-adapter_sd15.bin"
)

In [ ]:
from diffusers.utils import load_image

source_image = load_image("/content/clipseg_source_image.png")
ip_image = load_image("/content/vermeer.png")

pipeline.to("cuda:0")

image = pipeline(
    prompt                   = 'best quality, high quality'
    , negative_prompt        = "monochrome,lowres, bad anatomy,low quality"
    , image                  = source_image
    , ip_adapter_image       = ip_image
    , num_images_per_prompt  = 1
    , num_inference_steps    = 50
    , strength               = 0.5
    , generator              = torch.Generator("cuda:0").manual_seed(1)
).images[0]

pipeline.to("cpu")
torch.cuda.empty_cache()
image